# 1.5 Personal chef (capstone)

This notebook combines Module 1 pieces into one agent:

| Piece | Where it shows up |
| --- | --- |
| **System prompt** | Role: leftover-ingredient chef |
| **Tool** | Tavily `web_search` for real recipes |
| **Memory** | `InMemorySaver` + `thread_id` so follow-ups keep context |

You give ingredients; the agent should search and suggest dishes. Ask for full instructions in a later turn on the same thread.

Needs `OPENAI_API_KEY` and `TAVILY_API_KEY` in `.env`.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Web search tool

Same Tavily wrapper as the web-search notebook. The chef prompt tells the model to use this tool instead of inventing recipes from memory alone.

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

## System prompt

Spell out **role**, **when to call the tool**, and **what to return**. Vague prompts lead to skipped searches or walls of unusable text.

In [ ]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

## Create the agent

`tools`, `system_prompt`, and `checkpointer` together: searchable, in-character, and stateful across turns.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()  # keep this chat's history
)

## First turn

Reuse the same `config` / `thread_id` if you add a follow-up (for example "Give me the full instructions for the first one").

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

### Message trace

Full list should include tool calls and Tavily results, not only the final paragraph.

In [ ]:
from pprint import pprint

pprint(response)